# Text Summarization

Extractive systems tell you what the document said.
Abstractive systems tell you what the author meant.
Different tasks, different pitfalls.

## Problem definition

**Extractive:** Pick the three most important sentences from the article.

* Rank problem

**Abstractive:** Rewrite the content in your own words.

* Gneration problem

## Basic Concept

### Extractive

**TextRank.** Treat the article as a graph where nodes are sentences and edges are similarities. Run PageRank over the graph to score sentences by how connected they are to everything else. Highest-scoring sentences are the summary.

### Abstractive

**Pegasus.** Fine-tune a trainsformer encoder-decoder on document-summary pairs. At inference, the model reads the document and generates the summary token-by-otken via cross-attention.

### Evaluation

Recall-Oriented Understudy fro Gisting Evaluation (ROUGE)

* ROUGE-1,ROUGE-2. score unigram and bigram overlap
* ROUGE-L. score longest common subsequence



# Build your Own

## TextRank

In [2]:
import math
import re
from collections import Counter

def sentence_split(text):
    return re.split(r"(?<=[.!?])\s+", text.strip())

def similarity(s1, s2):
    w1 = Counter(s1.lower().split())
    w2 = Counter(s2.lower().split())
    intersection = sum((w1 & w2).values())
    denom = math.log(len(w1) + 1) + math.log(len(w2) + 1)
    if denom == 0:
        return 0.0
    return intersection / denom

def text_rank(text, top_k=3, damping=0.85, iterations=50, epsilon=1e-4):
    sentences = sentence_split(text)
    n = len(sentences)
    if n <= top_k:
        return sentences

    sim = [[0.0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            if i != j:
                sim[i][j] = similarity(sentences[i], sentences[j])

    scores = [1.0] * n
    for _ in range(iterations):
        new_scores = [1 - damping] * n
        for i in range(n):
            total_out = sum(sim[i]) or 1e-9
            for j in range(n):
                if sim[i][j] > 0:
                    new_scores[j] += damping * sim[i][j] / total_out * scores[i]
        if max(abs(s - ns) for s, ns in zip(scores, new_scores)) < epsilon:
            break
        scores = new_scores

    ranked = sorted(range(n), key=lambda i: scores[i], reverse=True)[:top_k]
    ranked.sort()
    return [sentences[i] for i in ranked]

TEST_TEXT = """
On the island of Blackthorn, where the sea met the sky in a grey, unbroken line, Elias Thorn kept the lighthouse. He had done so for forty years, and in that time, he had seen storms that could swallow ships whole and dawns so quiet they felt like forgiveness.

The lighthouse was not merely a tower of stone and glass. To Elias, it was a promise. Every evening at dusk, he climbed the spiral stairs, his knees protesting like old friends, and lit the great lamp. Its beam swept across the water in a slow, patient arc—once, twice, three times—before settling into its eternal rhythm. Sailors far out at sea would see that light and know they were not alone.

One winter, the supply boat stopped coming. Elias waited three weeks. Then four. The radio crackled only with static. He rationed his stores, caught fish from the rocks, and kept the light burning. What else was there to do?

On the forty-second night alone, he saw something strange from the gallery. A ship, black against the black water, moving without sails, without sound. It did not appear on any chart he had ever studied. Elias watched it approach until it vanished into the fog, as if the sea had simply swallowed it whole.

He told no one. There was no one to tell.

Years passed—or perhaps they did not. Time on Blackthorn moved differently, like water around a stone. Elias grew old. His hair turned the colour of sea foam. His hands, once steady enough to polish the lens to perfection, began to tremble. Still, each night, he climbed the stairs.

On what he knew would be his last evening, he reached the lamp room and found the wick already lit. The flame burned bright and clear, though he had not touched it. He sat on the cold stone step and watched the beam turn.

A voice came from below—not from the stairs, but from everywhere at once.

"You kept the promise," it said.

Elias smiled. "Someone had to."

The light continued its sweep. The sea breathed against the rocks. And somewhere far out, a sailor who had long since given up hope saw a beam cut through the darkness and turned his ship toward home.

When the coast guard finally reached Blackthorn, they found the lighthouse empty. The lamp was cold. But in the logbook, on the last page, Elias had written a single line:

The light was never for me. It was always for them.
"""

print(text_rank(TEST_TEXT))

['On the island of Blackthorn, where the sea met the sky in a grey, unbroken line, Elias Thorn kept the lighthouse.', 'On what he knew would be his last evening, he reached the lamp room and found the wick already lit.', 'But in the logbook, on the last page, Elias had written a single line:\n\nThe light was never for me.']


## Abstractive with BART

In [5]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

inputs = tokenizer(TEST_TEXT, max_length=1024, truncation=True, return_tensors="pt")
outputs = model.generate(
    **inputs,
    max_length=130,
    min_length=40,
    do_sample=False,
)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Elias Thorn kept the lighthouse on the island of Blackthorn for forty years. He rationed his stores, caught fish from the rocks, and kept the light burning. On the forty-second night alone, he saw something strange from the gallery.


## ROUGE evaluation

In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"], use_stemmer=True
)

summary_from_rank = "".join(text_rank(TEST_TEXT))
summary_from_bart = tokenizer.decode(outputs[0], skip_special_tokens=True)

scores = scorer.score(summary_from_rank, summary_from_bart)
print({k: round(v.fmeasure, 4) for k, v in scores.items()})